# Gamelog pipeline — Bronze → Silver

**Bronze** (`raw.*` in Supabase): fetch from NBA API → 5 tables  
**Silver** (`silver.player_gamelogs`): merge + positions + name canon + rotowire → upload

Run cells top-to-bottom. Set `SEASON` and `SEASON_TYPE` once in the config cell.

In [1]:
import os
import sys
from pathlib import Path

import pandas as pd

project_root = Path.cwd().parent
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

pd.set_option("display.max_columns", None)

In [37]:
# ── change these ──────────────────────────────────────────────────────────
SEASON = "2019-20"
SEASON_TYPE = "Playoffs"          # "Regular Season" | "Playoffs"
SKIP_ROTOWIRE = False             # True if you don't have the rotowire CSV yet
RUN_TRACKING_FETCH = False        # True to (re)fetch start_positions (slow)
UPLOAD_TO_SUPABASE = True         # False to build silver in memory only
# ─────────────────────────────────────────────────────────────────────────

## 1. Bronze — fetch NBA API → `raw.*`

| Function | Table | Source |
|---|---|---|
| `fetch_all(season, season_type)` | all 4 below | one call |
| `fetch_player_base` | `raw.player_base` | PlayerGameLogs Base |
| `fetch_player_adv` | `raw.player_adv` | PlayerGameLogs Advanced |
| `fetch_team_base` | `raw.team_base` | TeamGameLogs Base |
| `fetch_team_adv` | `raw.team_adv` | TeamGameLogs Advanced |
| `fetch_boxscoreplayertrackv3` | `raw.start_positions` | per-game tracking (slow) |

Requires `SUPABASE_DB_URL` in `.env`. Safe to re-run — upserts on natural keys.

In [38]:
from src.utils.bronze import fetch_all, fetch_boxscoreplayertrackv3

print(f"Bronze: {SEASON} {SEASON_TYPE}")
fetch_all(SEASON, SEASON_TYPE)

if RUN_TRACKING_FETCH:
    fetch_boxscoreplayertrackv3(
        SEASON,
        SEASON_TYPE,
        batch_size=100,
        delay=2.5,
        workers=5,
    )

Bronze: 2019-20 Playoffs
    … 1,694/1,694 rows
  ✓ raw.player_base — 1,694 rows upserted (postgres)
    … 1,694/1,694 rows
  ✓ raw.player_adv — 1,694 rows upserted (postgres)
    … 166/166 rows
  ✓ raw.team_base — 166 rows upserted (postgres)
    … 166/166 rows
  ✓ raw.team_adv — 166 rows upserted (postgres)


In [39]:
from src.utils.db import read_df

game_filter = "002%" if SEASON_TYPE == "Regular Season" else "004%"
params = {"season": SEASON, "prefix": game_filter}
where = "season_year = %(season)s AND game_id LIKE %(prefix)s"
sp_where = (
    "game_id IN (SELECT DISTINCT game_id FROM raw.player_base "
    f"WHERE season_year = %(season)s AND game_id LIKE %(prefix)s)"
)

print(f"{'table':<18} rows")
print("-" * 26)
for table in ("player_base", "player_adv", "team_base", "team_adv", "start_positions"):
    w = sp_where if table == "start_positions" else where
    n = len(read_df(table, where=w, params=params))
    flag = "  ⚠ empty" if table == "team_adv" and n == 0 else ""
    print(f"{table:<18} {n:>6,}{flag}")

table              rows
--------------------------
player_base         1,694
player_adv          1,694
team_base             166
team_adv              166
start_positions     2,119


### Bronze patch — team_adv only

Run this if verify shows `team_adv` at 0 rows (common for older playoff loads).

In [40]:
from src.utils.bronze import fetch_team_adv

fetch_team_adv(SEASON, SEASON_TYPE)

    … 166/166 rows
  ✓ raw.team_adv — 166 rows upserted (postgres)


## 2. Silver — merge `raw.*` → `silver.player_gamelogs`

| Function | What it does |
|---|---|
| `build_gamelogs_silver(season, season_type)` | read bronze + positions + name canon + rotowire |
| `upsert_silver(df, season_type=...)` | upload to `silver.player_gamelogs` |

- `IS_PLAYOFF` is set automatically from `season_type`
- Re-upsert updates existing rows on `(game_id, player_id)` — no need to drop the table

In [41]:
from importlib import reload
import src.utils.silver as silver
import src.utils.db as db
reload(silver)
reload(db)

from src.utils.silver import build_gamelogs_silver
from src.utils.db import upsert_silver

df = build_gamelogs_silver(
    SEASON,
    SEASON_TYPE,
    skip_rotowire=SKIP_ROTOWIRE,
)

df[["PLAYER_NAME", "GAME_DATE", "MATCHUP", "PTS", "TEAM_SPREAD", "GAME_TOTAL", "IS_PLAYOFF"]].head()

── Silver: 2019-20 Playoffs ──
  loading raw.* from Supabase…
  read raw.player_base — 1,694 rows
  read raw.player_adv — 1,694 rows
  read raw.team_base — 166 rows
  read raw.team_adv — 166 rows
  read raw.start_positions — 2,119 rows
  merged — (1694, 183)
  names — 217 unique | missing vs reference: 137
  missing sample: ['Abdel Nader', 'Alec Burks', 'Alize Johnson', 'Andre Iguodala', 'Andre Roberson', 'Antonius Cleveland', 'Austin Rivers', 'BJ Johnson', 'Ben McLemore', 'Boban Marjanovic', 'Bol Bol', 'Brad Wanamaker', 'Bruno Caboclo', 'Carmelo Anthony', 'Carsen Edwards', 'Chris Chiozza', 'Chris Clemons', 'D.J. Augustin', 'Damian Lillard', 'Daniel Theis']
✓ Silver frame — 1,694 rows, 188 columns


,PLAYER_NAME,GAME_DATE,MATCHUP,PTS,TEAM_SPREAD,GAME_TOTAL,IS_PLAYOFF
0,LeBron James,2020-10-11,LAL @ MIA,28.0,NaN,NaN,1
1,Bam Adebayo,2020-10-11,MIA vs. LAL,25.0,NaN,NaN,1
2,Anthony Davis,2020-10-11,LAL @ MIA,19.0,NaN,NaN,1
3,Rajon Rondo,2020-10-11,LAL @ MIA,19.0,NaN,NaN,1
4,Jimmy Butler,2020-10-11,MIA vs. LAL,12.0,NaN,NaN,1


In [42]:
if UPLOAD_TO_SUPABASE:
    upsert_silver(df, season_type=SEASON_TYPE)
else:
    print(f"Skipping upload — {len(df):,} rows in memory")

    … 1,694/1,694 rows
  ✓ silver.player_gamelogs — 1,694 rows upserted (postgres)


In [43]:
from src.utils.db import read_df

check = read_df(
    "player_gamelogs",
    schema="silver",
    where="season_year = %(s)s AND season_type = %(st)s",
    params={"s": SEASON, "st": SEASON_TYPE},
)
print(f"silver.player_gamelogs — {len(check):,} rows")
print(check["is_playoff"].value_counts())
check[["player_name", "game_date", "pts", "team_spread", "is_playoff"]].head()

silver.player_gamelogs — 1,694 rows
is_playoff
1    1694
Name: count, dtype: int64


,player_name,game_date,pts,team_spread,is_playoff
0,LeBron James,2020-10-11,28.0,None,1
1,Bam Adebayo,2020-10-11,25.0,None,1
2,Anthony Davis,2020-10-11,19.0,None,1
3,Rajon Rondo,2020-10-11,19.0,None,1
4,Jimmy Butler,2020-10-11,12.0,None,1
